# Machine Learning Capstone Project
## Regression Track: SVR, KNN & Random Forest Regressor
**Dataset:** WHO Life Expectancy Data (`Life Expectancy Data.csv`)  
**Target Variable:** `Life expectancy` (Continuous)  

---
### Notebook Structure
1. **Dataset Loading & Audit:** Data shape, schema, missing value detection, and target summary statistics.
2. **Exploratory Data Analysis (EDA):** Target distribution, feature distributions, correlation heatmaps, and scatter plots.
3. **Data Preprocessing & Feature Engineering:** Missing value imputation, outlier handling, train-only scaling, categorical encoding, and interaction features.
4. **Model Training & Hyperparameter Tuning:**
   * K-Nearest Neighbors (KNN) Regressor
   * Support Vector Regressor (SVR)
   * Random Forest Regressor (`GridSearchCV`)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor

import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 6)
RANDOM_STATE = 42


## 1. Dataset Loading & Audit

In [ ]:
df_raw = pd.read_csv('../data/Life Expectancy Data.csv')
df_raw.columns = df_raw.columns.str.strip()

print("="*60)
print(f"Dataset Shape: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns")
print("="*60)

audit_df = pd.DataFrame({
    'Data Type': df_raw.dtypes,
    'Null Count': df_raw.isnull().sum(),
    'Null Percentage (%)': (df_raw.isnull().sum() / len(df_raw)) * 100
})
display(audit_df[audit_df['Null Count'] > 0].sort_values(by='Null Count', ascending=False))


Dataset Shape: 2938 rows, 22 columns
Target Variable ('Life expectancy') Summary Statistics:
count    2928.000000
mean       69.224932
std         9.523867
min        36.300000
25%        63.100000
50%        72.100000
75%        75.700000
max        89.000000
Name: Life expectancy, dtype: float64


## 2. Exploratory Data Analysis (EDA)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df_raw['Life expectancy'], kde=True, color='teal', ax=ax[0])
ax[0].set_title('Target Distribution: Life Expectancy', fontsize=12, fontweight='bold')

sns.boxplot(x=df_raw['Life expectancy'], color='lightseagreen', ax=ax[1])
ax[1].set_title('Target Boxplot: Life Expectancy', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


## 3. Data Preprocessing & Feature Engineering

In [ ]:
df = df_raw.dropna(subset=['Life expectancy']).copy()
num_cols = df.select_dtypes(include=[np.number]).columns
for col in num_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df.groupby('Status')[col].transform(lambda x: x.fillna(x.median()))
        df[col] = df[col].fillna(df[col].median())

df['Status_Code'] = (df['Status'] == 'Developed').astype(int)
df['Schooling_Income_Index'] = df['Schooling'] * df['Income composition of resources']

features = [c for c in num_cols if c not in ['Life expectancy', 'Year']] + ['Status_Code', 'Schooling_Income_Index']
X = df[features]
y = df['Life expectancy']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print(f"Training shape: {X_train_scaled.shape} | Test shape: {X_test_scaled.shape}")


Training shape: (2342, 21) | Test shape: (586, 21)
